In [2]:
# ============================================================
# 0. Install dependencies (Colab)
# ============================================================
!pip install -q transformers peft accelerate z3-solver

# ============================================================
# 1. Imports
# ============================================================
import json, re, ast, math, torch
from dataclasses import dataclass
from typing import Callable, Dict, Any, List, Optional

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from z3 import Int, Solver, Distinct, Abs, sat

# Optional: Colab secret for HF token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None  # set manually if needed, e.g. "hf_..."

# ============================================================
# 2. TOOL SYSTEM (calculator + logic solver)
# ============================================================

@dataclass
class Tool:
    name: str
    description: str
    args_schema: Dict[str, Any]
    func: Callable[[Dict[str, Any]], Dict[str, Any]]


# --------- Safe Calculator ---------

class SafeEvaluator(ast.NodeVisitor):
    allowed_binops = {
        ast.Add, ast.Sub, ast.Mult, ast.Div,
        ast.FloorDiv, ast.Pow, ast.Mod
    }
    allowed_unaryops = {ast.UAdd, ast.USub}
    allowed_names = {"pi": math.pi, "e": math.e}
    allowed_funcs = {
        "sqrt": math.sqrt, "log": math.log, "ln": math.log,
        "sin": math.sin, "cos": math.cos, "tan": math.tan
    }

    def visit_Expression(self, node):
        return self.visit(node.body)

    def visit_BinOp(self, node):
        if type(node.op) not in self.allowed_binops:
            raise ValueError("Unsupported operator")
        l, r = self.visit(node.left), self.visit(node.right)
        return {
            ast.Add:  l + r,
            ast.Sub:  l - r,
            ast.Mult: l * r,
            ast.Div:  l / r,
            ast.FloorDiv: l // r,
            ast.Mod:  l % r,
            ast.Pow:  l ** r,
        }[type(node.op)]

    def visit_UnaryOp(self, node):
        if type(node.op) not in self.allowed_unaryops:
            raise ValueError("Unsupported unary operator")
        v = self.visit(node.operand)
        return +v if isinstance(node.op, ast.UAdd) else -v

    def visit_Name(self, node):
        if node.id not in self.allowed_names:
            raise ValueError(f"Name {node.id} not allowed")
        return self.allowed_names[node.id]

    def visit_Call(self, node):
        if not isinstance(node.func, ast.Name):
            raise ValueError("Only simple function calls allowed")
        if node.func.id not in self.allowed_funcs:
            raise ValueError(f"Function {node.func.id} not allowed")
        if node.keywords:
            raise ValueError("Keyword args not allowed")
        return self.allowed_funcs[node.func.id](
            *[self.visit(a) for a in node.args]
        )

    def visit_Constant(self, node):
        if not isinstance(node.value, (int, float)):
            raise ValueError("Only int/float constants allowed")
        return node.value

    def visit_Num(self, node):  # for older Python
        return node.n

    def generic_visit(self, node):
        raise ValueError(f"Unsupported syntax: {type(node).__name__}")


def safe_eval_expression(expr: str) -> float:
    tree = ast.parse(expr, mode="eval")
    return SafeEvaluator().visit(tree)


def calculator_tool_func(args: Dict[str, Any]) -> Dict[str, Any]:
    expr = args.get("expression")
    if not isinstance(expr, str) or not expr.strip():
        return {
            "status": "error",
            "expression": expr or "",
            "error": "Missing or invalid 'expression' argument.",
        }
    try:
        value = safe_eval_expression(expr)
        return {"status": "ok", "expression": expr, "result": value}
    except Exception as e:
        return {
            "status": "error",
            "expression": expr,
            "error": str(e),
        }


calculator_tool = Tool(
    name="calculator",
    description=(
        "Evaluate arithmetic expressions. Call exactly as:\n"
        '{"tool":"calculator","args":{"expression":"2*(3+4)"}}'
    ),
    args_schema={
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
    func=calculator_tool_func,
)


# --------- Constraint Solver ---------

def solve_row_csp(spec: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simplified solver, hard-wired to 3 houses (3x3 puzzles).

    Accepts either:
      - categories as a dict:
            {
              "name": ["Alice","Bob","Clara"],
              "music": ["Rock","Jazz","Pop"],
              ...
            }
        (old behavior)

      - OR categories as a list of category names:
            ["name","music","shirt_color",...]
        In that case we infer the actual values for each category from the
        constraint 'value'/'value1'/'value2' fields, and pad with dummy
        values so each category has exactly 3 distinct values.
    """
    houses = 3  # <--- hardwired to 3x3

    raw_categories = spec["categories"]
    constraints = spec["constraints"]

    # --- Normalize categories into a dict[category -> list of values] ---

    if isinstance(raw_categories, dict):
        # Old style: user/model provided full mapping
        categories = raw_categories
    elif isinstance(raw_categories, list):
        # New style: list of category names -> infer values from constraints
        if not all(isinstance(c, str) for c in raw_categories):
            raise ValueError(
                "'categories' list must contain only strings (category names)."
            )

        categories: Dict[str, List[str]] = {}

        # helper: collect all values for a given category from constraints
        def collect_vals_for(cat: str) -> List[str]:
            vals = set()
            for c in constraints:
                if not isinstance(c, dict):
                    continue

                t = c.get("type")

                # position / not_position use single 'category' / 'value'
                if t in {"position", "not_position"}:
                    if c.get("category") == cat and "value" in c:
                        vals.add(str(c["value"]))

                # binary constraints use category1/value1 and category2/value2
                if t in {
                    "same_house",
                    "not_same_house",
                    "next_to",
                    "immediately_left_of",
                    "immediately_right_of",
                    "somewhere_left_of",
                    "somewhere_right_of",
                }:
                    if c.get("category1") == cat and "value1" in c:
                        vals.add(str(c["value1"]))
                    if c.get("category2") == cat and "value2" in c:
                        vals.add(str(c["value2"]))

            vals = list(vals)
            if len(vals) > houses:
                raise ValueError(
                    f"Category {cat!r} has more than {houses} distinct values "
                    f"in constraints: {vals}"
                )

            # Pad with dummy values if we don't have enough
            k = 1
            while len(vals) < houses:
                dummy = f"{cat}_dummy_{k}"
                if dummy not in vals:
                    vals.append(dummy)
                k += 1
            return vals

        for cat in raw_categories:
            categories[cat] = collect_vals_for(cat)
    else:
        raise ValueError(
            "'categories' must be either an object (old style) or a list of "
            "category names (new style)."
        )

    # --- Category validation (still enforced) ---
    for cat, vals in categories.items():
        if len(vals) != len(set(vals)):
            raise ValueError(f"Category {cat!r} has duplicate values: {vals}")
        if len(vals) != houses:
            raise ValueError(
                f"Category {cat!r} must have exactly {houses} distinct values, "
                f"got {len(vals)}: {vals}"
            )

    # --- Build Z3 variables ---
    solver = Solver()
    pos: Dict[tuple, Any] = {}

    for cat, vals in categories.items():
        vars_for_cat = []
        for v in vals:
            safe_val = re.sub(r"[^a-zA-Z0-9_]", "_", str(v))
            name = f"pos_{cat}_{safe_val}"
            var = Int(name)
            pos[(cat, v)] = var
            solver.add(var >= 1, var <= houses)
            vars_for_cat.append(var)
        solver.add(Distinct(vars_for_cat))

    def var(cat: str, val: str):
        return pos[(cat, val)]

    # --- Add constraints with strict schema ---
    for c in constraints:
        if not isinstance(c, dict):
            raise ValueError(f"Constraint must be an object, got: {c!r}")
        if "type" not in c:
            raise ValueError(f"Constraint missing 'type' field: {c!r}")

        t = c["type"]

        if t == "same_house":
            solver.add(
                var(c["category1"], c["value1"])
                == var(c["category2"], c["value2"])
            )
        elif t == "not_same_house":
            solver.add(
                var(c["category1"], c["value1"])
                != var(c["category2"], c["value2"])
            )
        elif t == "position":
            solver.add(
                var(c["category"], c["value"]) == int(c["position"])
            )
        elif t == "not_position":
            solver.add(
                var(c["category"], c["value"]) != int(c["position"])
            )
        elif t == "next_to":
            solver.add(
                Abs(
                    var(c["category1"], c["value1"])
                    - var(c["category2"], c["value2"])
                ) == 1
            )
        elif t == "immediately_left_of":
            solver.add(
                var(c["category1"], c["value1"]) + 1
                == var(c["category2"], c["value2"])
            )
        elif t == "immediately_right_of":
            solver.add(
                var(c["category1"], c["value1"])
                == var(c["category2"], c["value2"]) + 1
            )
        elif t == "somewhere_left_of":
            solver.add(
                var(c["category1"], c["value1"])
                < var(c["category2"], c["value2"])
            )
        elif t == "somewhere_right_of":
            solver.add(
                var(c["category1"], c["value1"])
                > var(c["category2"], c["value2"])
            )
        else:
            raise ValueError(f"Unknown constraint type: {t!r}")

    # --- Solve ---
    if solver.check() != sat:
        return {"status": "unsat", "solution": None}

    model = solver.model()
    solution: Dict[str, Dict[str, int]] = {}
    for cat, vals in categories.items():
        solution[cat] = {v: model[pos[(cat, v)]].as_long() for v in vals}

    return {"status": "sat", "solution": solution}


def logic_csp_tool_func(args: Dict[str, Any]) -> Dict[str, Any]:
    try:
        return solve_row_csp(args)
    except Exception as e:
        return {"status": "error", "error": str(e)}


logic_csp_tool = Tool(
    name="logic_csp_solver",
    description=(
        "Solve 3x3 (3 houses) Zebra/Einstein puzzles with 3 positions.\n\n"
        "Input JSON MUST have:\n"
        "  - 'houses': (ignored; solver always uses 3)\n"
        "  - 'categories': EITHER:\n"
        "       * an object mapping category name -> EXACTLY 3 DISTINCT values, e.g.\n"
        '         {"name":["Alice","Bob","Clara"], ...}\n'
        "       * OR a list of category names like [\"name\",\"music\",\"shirt_color\"].\n"
        "         In that case, the solver infers the values for each category\n"
        "         from the constraints' 'value'/'value1'/'value2' fields and pads\n"
        "         with dummy values up to 3.\n"
        "  - 'constraints': array of constraint OBJECTS, each with a 'type' field.\n\n"
        "Allowed constraint types:\n"
        "  - same_house: {\"type\":\"same_house\", \"category1\":..., \"value1\":..., \"category2\":..., \"value2\":...}\n"
        "  - not_same_house: {\"type\":\"not_same_house\", ...}\n"
        "  - position: {\"type\":\"position\", \"category\":..., \"value\":..., \"position\":1..3}\n"
        "  - not_position: {\"type\":\"not_position\", ...}\n"
        "  - next_to: {\"type\":\"next_to\", \"category1\":..., \"value1\":..., \"category2\":..., \"value2\":...}\n"
        "  - immediately_left_of: {\"type\":\"immediately_left_of\", ...}\n"
        "  - immediately_right_of: {\"type\":\"immediately_right_of\", ...}\n"
        "  - somewhere_left_of: {\"type\":\"somewhere_left_of\", ...}\n"
        "  - somewhere_right_of: {\"type\":\"somewhere_right_of\", ...}\n\n"
        "IMPORTANT: Do NOT invent new 'type' values. Do NOT use plain strings in 'constraints'.\n"
        "This tool is optimized for 3x3 puzzles; 'houses' is always treated as 3."
    ),
    args_schema={"type": "object"},
    func=logic_csp_tool_func,
)

# Stricter schema hint (key names made explicit)
LOGIC_CSP_SCHEMA_HINT = """
For the logic_csp_solver tool:

- 'categories' must map each category name to EXACTLY N distinct values (N = 'houses').

Constraint schemas (use EXACTLY these keys):

- same_house:
  {"type":"same_house",
   "category1":"<catA>", "value1":"<valA>",
   "category2":"<catB>", "value2":"<valB>"}

- not_same_house:
  {"type":"not_same_house",
   "category1":"<catA>", "value1":"<valA>",
   "category2":"<catB>", "value2":"<valB>"}

- position:
  {"type":"position",
   "category":"<category_name>",
   "value":"<value>",
   "position": <integer 1..N>}

- not_position:
  {"type":"not_position",
   "category":"<category_name>",
   "value":"<value>",
   "position": <integer 1..N>}

- next_to / immediately_left_of / immediately_right_of /
  somewhere_left_of / somewhere_right_of:
  always use "category1","value1","category2","value2", for example:

  {"type":"immediately_right_of",
   "category1":"name", "value1":"Tiffany",
   "category2":"accessory", "value2":"ring"}

Do NOT invent other keys like 'house' instead of 'position'.
Do NOT use 'category' + 'category2' where the schema requires 'category1' + 'category2'.

If the tool returns an error about the schema, FIX the JSON and call the tool again.
"""

# ============================================================
# 3. Strict JSON extractor
# ============================================================

def extract_json(output: str) -> Dict[str, Any]:
    s = output.strip()
    try:
        start = s.index("{")
        end = s.rindex("}") + 1
        return json.loads(s[start:end])
    except Exception:
        raise ValueError(f"Could not parse JSON from:\n{output}")


# ============================================================
# 4. Tool Agent – with nested-tool handling & retry
# ============================================================

BASE_SYSTEM_PROMPT = """
You are a tool-using assistant.

You MUST ALWAYS respond with exactly ONE JSON object and NOTHING else.

VALID OUTPUT FORMS:

1) Tool call:
   {"tool": "<name>", "args": {...}}

   For the tools you have:

   - calculator:
       Purpose: evaluate arithmetic expressions.
       You MUST call it exactly as:
       {"tool": "calculator", "args": {"expression": "<math expression as a string>"}}

   - logic_csp_solver:
       Purpose: solve row-based Zebra/Einstein puzzles.
       Input MUST respect the schema described in the tool description.
       Only use the allowed 'type' values listed there.
       Do NOT put free-form strings in 'constraints'.

2) Final answer:
   {"final_answer": "<your explanation and answer in natural language>"}

NEVER write "User:" or "Tool:" in your output.
NEVER include any text outside the single JSON object.
NEVER mix tool calls with final_answer in the same object.
"""


class ToolAgent:
    def __init__(
        self,
        model,
        tokenizer,
        tools: List[Tool],
        max_tool_calls: int = 1,
        max_new_tokens: int = 256,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.tools = {t.name: t for t in tools}
        self.max_tool_calls = max_tool_calls
        self.max_new_tokens = max_new_tokens

    def _gen(self, system_prompt: str, user_block: str) -> str:
        """
        Build a single [INST] block:
        [INST] <<SYS>> system_prompt <</SYS>> user_block [/INST]
        """
        prompt = f"[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_block} [/INST]"
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,  # deterministic
                pad_token_id=self.tokenizer.eos_token_id,
            )
        gen = out[0, inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(gen, skip_special_tokens=True)

    def _maybe_unwrap_nested_tool(self, obj: Dict[str, Any]) -> Dict[str, Any]:
        """
        If the model returned something like:
          {"final_answer": "{\"tool\": \"logic_csp_solver\", ...}"}
        then unwrap the inner JSON and treat it as the real object.
        """
        fa = obj.get("final_answer")
        if isinstance(fa, str):
            inner = fa.strip()
            # cheap heuristic: looks like a JSON object with a "tool" key
            if "{" in inner and '"tool"' in inner:
                try:
                    inner_obj = extract_json(inner)
                    if isinstance(inner_obj, dict) and "tool" in inner_obj:
                        return inner_obj
                except Exception:
                    # we *don't* swallow this; caller may decide to retry
                    pass
        return obj

    def generate(self, user_query: str, extra_system_prompt: Optional[str] = None):
        system_prompt = BASE_SYSTEM_PROMPT
        if extra_system_prompt:
            system_prompt += "\n" + extra_system_prompt

        context = ""  # describes tool calls & results in plain text
        tool_calls: List[Dict[str, Any]] = []
        tool_traces: List[Dict[str, Any]] = []

        for _ in range(self.max_tool_calls):
            user_block = (
                "User question:\n"
                f"{user_query}\n\n"
                "Context (results from tools so far):\n"
                f"{context}\n\n"
                "Now decide your next move.\n"
                "If you need to use a tool, respond with exactly one JSON object of the form:\n"
                '{"tool": "<name>", "args": {...}}\n'
                "Otherwise, if you can answer the user, respond with exactly one JSON object of the form:\n"
                '{"final_answer": "<text>"}\n'
                "Remember: no extra text, no explanations outside the JSON.\n"
            )

            raw = self._gen(system_prompt, user_block)

            try:
                obj = extract_json(raw)
            except Exception:
                # Couldn't parse at all -> bail out with raw text
                return {
                    "final_answer": raw,
                    "tool_calls": tool_calls,
                    "tool_traces": tool_traces,
                }

            # First try to unwrap a *valid* nested tool JSON
            unwrapped = self._maybe_unwrap_nested_tool(obj)

            if unwrapped is not obj and "tool" in unwrapped:
                obj = unwrapped
            else:
                # If this is a final_answer that *looks* like a tool call but failed to unwrap,
                # treat it as an error and retry instead of stopping.
                fa = obj.get("final_answer")
                if (
                    "final_answer" in obj
                    and "tool" not in obj
                    and isinstance(fa, str)
                    and '"tool"' in fa
                    and "{" in fa
                ):
                    # Add an error message into context and try another step
                    error_msg = {
                        "status": "error",
                        "error": (
                            "Model produced a nested tool JSON inside 'final_answer' "
                            "that was invalid or truncated. "
                            "Please output a DIRECT tool call JSON of the form "
                            '{"tool": "...", "args": {...}} with valid, compact JSON.'
                        )
                    }
                    context += (
                        "\nTool 'logic_csp_solver' result:\n"
                        + json.dumps(error_msg, indent=2, ensure_ascii=False)
                        + "\n"
                    )
                    # Continue the for-loop -> re-prompt the model
                    continue

            # Final answer path (normal case)
            if "final_answer" in obj and "tool" not in obj:
                return {
                    "final_answer": obj["final_answer"],
                    "tool_calls": tool_calls,
                    "tool_traces": tool_traces,
                }

            # Tool call path
            if "tool" in obj:
                name = obj["tool"]
                args = obj.get("args", {}) or {}
                tool_calls.append({"tool": name, "args": args})

                tool = self.tools.get(name)
                if tool is None:
                    result = {
                        "status": "error",
                        "error": f"Unknown tool: {name}",
                    }
                else:
                    result = tool.func(args)

                tool_traces.append(
                    {"tool": name, "args": args, "result": result}
                )

                context += (
                    f"\nTool '{name}' result:\n"
                    f"{json.dumps(result, indent=2, ensure_ascii=False)}\n"
                )
                continue

            # Unexpected JSON shape → treat whole raw string as final answer
            return {
                "final_answer": raw,
                "tool_calls": tool_calls,
                "tool_traces": tool_traces,
            }

        # If we used up max_tool_calls, force a final answer
        final_user_block = (
            "User question:\n"
            f"{user_query}\n\n"
            "Context (results from tools so far):\n"
            f"{context}\n\n"
            "You have already used the allowed number of tool calls.\n"
            "Now you MUST respond only with a final answer JSON of the form:\n"
            '{"final_answer": "<text>"}\n'
            "No other keys, no extra text.\n"
        )
        raw = self._gen(system_prompt, final_user_block)
        try:
            obj = extract_json(raw)
            obj = self._maybe_unwrap_nested_tool(obj)
            final_answer = obj.get("final_answer", raw)
        except Exception:
            final_answer = raw

        return {
            "final_answer": final_answer,
            "tool_calls": tool_calls,
            "tool_traces": tool_traces,
        }


# ============================================================
# 5. Loader
# ============================================================

def load_llama(
    base_id: str,
    lora: Optional[str] = None,
    token: Optional[str] = None,
    device: Optional[str] = None,
):
    tokenizer = AutoTokenizer.from_pretrained(base_id, token=token)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    model = AutoModelForCausalLM.from_pretrained(
        base_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        token=token,
    )
    if lora:
        model = PeftModel.from_pretrained(model, lora)

    model.to(device)
    return model, tokenizer


# ============================================================
# 6. Config + Agent construction
# ============================================================

BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # <--- tunable
LORA_ADAPTER_ID = None                         # e.g. "path-or-hf-id-of-lora", or None

model, tokenizer = load_llama(BASE_MODEL_ID, lora=LORA_ADAPTER_ID, token=HF_TOKEN)
agent = ToolAgent(model, tokenizer, [calculator_tool, logic_csp_tool], max_tool_calls=2)

print("Agent READY.")

# ============================================================
# 7. Demo usage (4x4 zebra puzzle; accessory has 4 values conceptually)
# ============================================================

user_query = """
Find an ordering for a 4x4 zebra puzzle that satisfies the following constraints:

- There are 4 women in positions 1 to 4 (left to right).
- Each has a different name, favorite music genre, shirt color, and accessory.
- The accessories are: gloves, ring, bracelet, and scarf (even if not all are mentioned explicitly in the clues).

Constraints:
1. The woman wearing gloves is in the last position.
2. The woman who likes Country music is in the first position.
3. Tiffany is immediately after the woman wearing a ring.
4. The woman who likes Blues music is immediately before Tiffany.
5. Eleanor is in the third position.
6. The person wearing the Blue shirt is also wearing a bracelet.
7. The woman who likes Pop music is wearing a bracelet.
8. The person in the Red shirt is at one of the ends.
9. Regina likes Country music.
10. The woman in the Green shirt likes Soul music.

Use the logic_csp_solver tool to model this as a 1D zebra puzzle with:
- houses = 4
- categories: 'name', 'music', 'shirt_color', 'accessory'
- constraints in the EXACT schema supported by the tool.
"""

result = agent.generate(
    user_query,
    extra_system_prompt=LOGIC_CSP_SCHEMA_HINT,
)

print(json.dumps(result, indent=2, ensure_ascii=False))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 42.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Agent READY.
{
  "final_answer": " {\"tool\": \"logic_csp_solver\", \"args\": {\"houses\": 4, \"categories\": [\"name\", \"music\", \"shirt_color\", \"accessory\"], \"constraints\": [{\"type\": \"position\", \"category\": \"name\", \"value\": \"Eleanor\", \"position\": 3}, {\"type\": \"position\", \"category\": \"music\", \"value\": \"Country\", \"position\": 1}, {\"type\": \"same_house\", \"category1\": \"name\", \"value1\": \"Tiffany\", \"category2\": \"accessory\", \"value2\": \"ring\"}, {\"type\": \"immediately_right_of\", \"category1\": \"accessory\", \"value1\": \"ring\", \"category2\": \"name\", \"value2\": \"Tiffany\"}, {\"type\": \"immediately_left_of\", \"category1\": \"music\", \"value1\": \"Blues\", \"category2\": \"name\", \"value2\": \"Tiffany\"}, {\"type\": \"same_house\", \"category1\": \"shirt_color\", \"value1\": \"Blue\", \"category2\": \"accessory\", \"value2\": \"bracelet\"}, {\"type\": \"same_house\", \"category1\": \"music\", \"value",
  "tool_calls": [],
  "tool

In [3]:

# ============================================================
# 7. Demo usage (4x4 zebra puzzle; accessory has 4 values conceptually)
# ============================================================

user_query = """
produce an ordering for the followign 3x3 zebra puzzle. Use your calculator and solver tools where appropriate. Not all possible values are provided, invent a placeholder in that case.
 1) The person drinking milk lives in the yellow house.
2) The cats live directly to the right of the person drinking tea.
3) The mice live directly to the left of the person drinking espresso.
4) The person drinking milk lives directly to the left of the pink house."""

result = agent.generate(
    user_query,
    extra_system_prompt=LOGIC_CSP_SCHEMA_HINT,
)

print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "final_answer": " {\"final_answer\": \"The solution to the 3x3 zebra puzzle is as follows:\\n\\nHouse 1: Yellow house, drinks milk, has mice\\nHouse 2: Pink house, drinks tea, has cats\\nHouse 3: Blue house, drinks espresso, has other pet\\n\\nAll constraints are satisfied:\\n1) The person drinking milk lives in the yellow house. ✓\\n2) The cats live directly to the right of the person drinking tea. ✓ (Tea is in house 3, cats are in house 2, which is to the left — this appears to be a contradiction. Wait, correction: the cats are in house 2, tea is in house 3, so cats are to the left of tea, not right. This violates constraint 2. Let me recheck the logic.\\n\\nRe-evaluating constraint 2: 'The cats live directly to the right of the person drinking tea.' This means tea must be in a house immediately to the left of the cats.\\n\\nIn the current solution, tea is in house 3, cats are in house 2 — this is incorrect. The cats should be to the right of tea, so tea must be in house 1",
  "t

In [4]:
# ==========================================
# MODEL A: NEURO-SYMBOLIC AGENT (Qwen-4B + Z3)
# ==========================================
import gradio as gr
import json
import subprocess
from typing import Tuple

# 1. Install Gradio (if not already installed)
try:
    import gradio
except ImportError:
    print("🛠️ Installing Gradio UI...")
    subprocess.run("pip install -q gradio", shell=True)
    import gradio as gr

# 2. Define the Wrapper Function
def solve_model_a(user_text: str) -> Tuple[str, str, str]:
    """
    Runs the Neuro-Symbolic Agent (Global 'agent' object must be loaded).
    """
    if not user_text.strip():
        return "Please enter a puzzle.", "", ""

    # Access the pre-loaded agent from your previous code cells
    global agent
    # Use the schema hint if available, otherwise None
    hint = globals().get("LOGIC_CSP_SCHEMA_HINT", None)

    try:
        # Execute the Neuro-Symbolic Logic
        response = agent.generate(
            user_text,
            extra_system_prompt=hint
        )

        # 1. Final Answer (Natural Language)
        final_answer = response.get("final_answer", "No answer generated.")

        # 2. Tool Call (The JSON the model created)
        tool_calls = response.get("tool_calls", [])
        tool_call_str = json.dumps(tool_calls, indent=2, ensure_ascii=False) if tool_calls else "[]"

        # 3. Tool Trace (The actual execution result from Python/Z3)
        tool_traces = response.get("tool_traces", [])
        tool_trace_str = json.dumps(tool_traces, indent=2, ensure_ascii=False) if tool_traces else "[]"

        return final_answer, tool_call_str, tool_trace_str

    except Exception as e:
        return f"System Error: {str(e)}", "[]", "[]"

# 3. Create the Consistent Interface
demo = gr.Interface(
    fn=solve_model_a,
    inputs=gr.Textbox(
        lines=8,
        placeholder="Enter your math word problem or Zebra puzzle constraints here...",
        label="Input Query (Arithmetic or Logic Problem)"
    ),
    outputs=[
        gr.Textbox(label="1. Final Answer / Solution", lines=10),

    ],
    title="Model A",
    description="Please test the agent's ability to solve arithmetic word problems and complex logic puzzles",
    theme="soft"
)

print("🚀 Launching Model A UI...")
demo.launch(share=True, debug=True)

🚀 Launching Model A UI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9f21cae9b8995c6f62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9f21cae9b8995c6f62.gradio.live
